# 01 — Baseline MedGemma 4B

Charge une image, applique le prompt baseline via les prédictions MedGemma mises en cache
(inférence hors-ligne, 4-bit) et affiche la sortie JSON après garde-fous.

L'inférence lourde est exécutée une seule fois sur GPU par `eval/run_inference_batch.py` ;
ce notebook lit le cache et reste exécutable sur CPU.

In [1]:
from pathlib import Path
import sys, json
sys.path.append(str(Path('..').resolve()))
from src.inference import predict
from src.guardrails import apply_safety_guardrails

## Une prédiction unitaire
Cas d'opacité : la baseline la détecte prudemment (confiance modérée).

In [2]:
sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
result = apply_safety_guardrails(predict(sample, mode='baseline'))
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "image_quality": "good",
  "predicted_class": "suspected_opacity",
  "confidence": 0.7,
  "visual_evidence": [
    "Possible opacity in the lower lung field"
  ],
  "justification": "There is a possible opacity in the lower lung field, which could be consistent with pneumonia or other lung pathology. Further evaluation is needed to confirm the diagnosis.",
  "limitations": [
    "Limited view of the lung fields",
    "No clinical context"
  ],
  "warning": "Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.",
  "model_name": "google/medgemma-4b-it",
  "prompt_version": "baseline_v1",
  "latency_ms": 33,
  "guardrail_errors": []
}


## Vérification du contrat de sortie
Le schéma doit contenir la classe, la confiance, les observations, la justification, les
limites et l'avertissement obligatoire.

In [3]:
required = ['predicted_class','confidence','visual_evidence','justification',
            'limitations','warning']
assert all(k in result for k in required), 'schema incomplet'
assert result['predicted_class'] in ('normal','suspected_opacity','uncertain')
assert result['warning'], 'avertissement manquant'
print('Contrat de sortie respecté. Classe =', result['predicted_class'],
      '| confiance =', result['confidence'])

Contrat de sortie respecté. Classe = suspected_opacity | confiance = 0.7


## Baseline sur les 30 cas
Distribution des prédictions et exactitude brute (indicative : jeu synthétique).

In [4]:
recs = json.loads(Path('../eval/cached_predictions/predictions_baseline.json').read_text())
from collections import Counter
dist = Counter(r['prediction']['predicted_class'] for r in recs)
acc = sum(r['ground_truth']==r['prediction']['predicted_class'] for r in recs)/len(recs)
print('Distribution des prédictions :', dict(dist))
print(f'Exactitude baseline : {acc:.1%} (30 cas synthétiques)')
print('Note : score indicatif du pipeline, non une performance clinique.')

Distribution des prédictions : {'normal': 16, 'suspected_opacity': 4, 'uncertain': 10}
Exactitude baseline : 80.0% (30 cas synthétiques)
Note : score indicatif du pipeline, non une performance clinique.
